# 02 — 2D Thigh Segmentation, Optimal Reference Frame & 3-Point Anchor Selection

This notebook demonstrates **Stage 2** of the `monocular-3d-gait-tracking-deltav2` pipeline:
1. Selecting the optimal mid-stance reference frame via `ReferenceFrameFinder` (`src/preprocessing/vision/frame_selection.py`).
2. Extracting Canny edges and fitting anterior/posterior thigh boundary lines (`src/preprocessing/vision/edge_processing.py`, `line_fitting.py`).
3. Computing the well-conditioned 3-point anatomical thigh query triangle ($P_1, P_2, P_3$) via `ThighPointSelector` and `TriangleComposer`.

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'examples' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.vision.geometry import TriangleComposer
from src.preprocessing.vision.frame_selection import ReferenceFrameFinder
from src.preprocessing.vision.thigh_point_selector import ThighPointSelector

In [ ]:
# Load persisted 2D reference frames and anatomical thigh triplets from data/videos_state.json
state_path = PROJECT_ROOT / 'data/videos_state.json'
videos_state = json.loads(state_path.read_text(encoding='utf-8'))
sample_video = 'P27_E2T2_90.mp4'
entry = videos_state.get(sample_video, next(iter(videos_state.values())))
print(f'Video: {sample_video}')
print('  Walk Start Frame:', entry.get('walk_start'))
print('  Reference Frame :', entry.get('ref_frame'))
print('  Thigh Triplet 2D:', entry.get('points'))